# 04 — Mapa visual dels clients (Folium)

Verifica visualment que les coordenades dels 25 punts tenen sentit.
Genera un mapa HTML interactiu que pots obrir al navegador.

In [ ]:
import pandas as pd
import folium
from pathlib import Path

PROCESSED = Path('../data/processed')

# Carregar clients geocodificats
clients = pd.read_csv(PROCESSED / 'clients_geo.csv')
print(f'Punts carregats: {len(clients)}')

# Colors per zona
colors = {
    'Mollet del Vallès': 'black',
    'CALLDETENES': 'blue',
    'FOLGUEROLES': 'green',
    'SANT JULIÀ DE VILATORTA': 'purple',
    'VIC': 'red',
    'SANT FOST DE CAMPSENTELLES': 'orange',
}

# Crear mapa centrat a Vic
m = folium.Map(
    location=[41.93, 2.28],
    zoom_start=11,
    tiles='OpenStreetMap'
)

# Afegir marcador per cada client
for _, row in clients.iterrows():
    color = colors.get(row['poblacio'], 'gray')
    
    # Depot: marcador especial
    if row['client_id'] == 0:
        folium.Marker(
            location=[row['lat'], row['lon']],
            popup=f"<b>🏭 {row['nom']}</b><br>Punt de sortida i arribada",
            tooltip=row['nom'],
            icon=folium.Icon(color='black', icon='home', prefix='fa')
        ).add_to(m)
    else:
        idx = int(row.name)
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=8,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            popup=f"<b>{row['nom']}</b><br>{row['poblacio']}<br>Palets: {row['palets']}",
            tooltip=f"{idx}. {row['nom']}"
        ).add_to(m)

# Dibuixar la ruta baseline (ordre original)
ordre_original = list(range(len(clients))) + [0]  # 0→1→2→...→25→0
coords_ruta = [(clients.iloc[i]['lat'], clients.iloc[i]['lon']) for i in ordre_original]

folium.PolyLine(
    coords_ruta,
    color='gray',
    weight=2,
    opacity=0.5,
    tooltip='Ruta baseline (ordre actual)'
).add_to(m)

# Llegenda
llegenda = """
<div style='position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:10px; border-radius:8px;
     border:2px solid gray; font-size:13px;'>
<b>🍺 Damm Smart Truck</b><br>
Ruta DR0051 — 5/2/2026<br><br>
<span style='color:black'>⬤</span> DDI Mollet (Depot)<br>
<span style='color:blue'>⬤</span> Calldetenes (5)<br>
<span style='color:green'>⬤</span> Folgueroles (4)<br>
<span style='color:purple'>⬤</span> Sant Julià de Vilatorta (5)<br>
<span style='color:red'>⬤</span> Vic (10)<br>
<span style='color:orange'>⬤</span> Sant Fost (1)<br><br>
<span style='color:gray'>— —</span> Ruta baseline<br>
Total: 25 clients · 142 km · 9.8h
</div>
"""
m.get_root().html.add_child(folium.Element(llegenda))

# Guardar
OUT = PROCESSED / 'mapa_baseline.html'
m.save(str(OUT))
print(f'✓ Mapa guardat a {OUT}')
print('Obre el fitxer mapa_baseline.html al navegador per veure el mapa!')

## Com obrir el mapa

1. Ves a la carpeta `data/processed/`
2. Fes doble clic sobre `mapa_baseline.html`
3. S'obre al navegador — pots fer zoom, clicar cada punt, etc.

Verifica que:
- El depot (casa negra) és a Mollet del Vallès ✓
- Els punts vermells són al centre de Vic ✓  
- Els punts blaus són a Calldetenes (sud de Vic) ✓
- Els punts verds són a Folgueroles (est de Vic) ✓
- Els punts liles són a Sant Julià de Vilatorta (nord-est) ✓
- El punt taronja (412 Coffee Beer) és a Sant Fost, molt al sud ✓

Si algun punt és clarament fora de lloc, avisa per corregir-lo.